# 10｜Choice财务与分红 OpenBB 查询验收

本Notebook只读取公司SQLite数据库，通过`provider="qianji"`调用OpenBB，不登录Choice、不导入`EmQuantAPI`、不消耗CTR流量。

查询接口：利润表、资产负债表、现金流量表和历史现金分红。运行前请先应用0.9.0补丁并完成00号OpenBB环境构建。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE = ""  # Notebook放在项目notebooks目录时保持为空

def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"项目根目录不正确：{candidate}")
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "qianji_data_mini").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请填写PROJECT_ROOT_OVERRIDE。")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python路径：", sys.executable)
print("项目根目录：", PROJECT_ROOT)
print("输出目录：", OUTPUT_DIR)

Python路径： d:\minicoda3\envs\dm311\python.exe
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
输出目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output


In [2]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)
try:
    installed_qianji = version("qianji-data-mini")
except PackageNotFoundError:
    installed_qianji = "0.0.0"
try:
    installed_openbb = version("openbb")
except PackageNotFoundError:
    installed_openbb = "未安装"

version_ok = Version(installed_qianji) >= Version("0.9.0")
print("qianji-data-mini：", installed_qianji)
print("OpenBB：", installed_openbb)
if not version_ok:
    raise RuntimeError("当前版本低于0.9.0，请运行00_openBB环境构建.ipynb后重启内核。")

qianji-data-mini： 0.9.0
OpenBB： 4.7.2


In [3]:
SYMBOLS = ["000001.SZ", "600519.SH", "300750.SZ"]
START_REPORT_DATE = "2025-01-01"
END_REPORT_DATE = "2026-12-31"
LIMIT = 10
STRICT_MODE = False

print("查询证券：", SYMBOLS)
print("报告期范围：", START_REPORT_DATE, "至", END_REPORT_DATE)

查询证券： ['000001.SZ', '600519.SH', '300750.SZ']
报告期范围： 2025-01-01 至 2026-12-31


In [4]:
import sqlite3
import pandas as pd
from IPython.display import display
from qianji_data_mini import Database

database = Database()
DB_PATH = database.path
with database.connect() as connection:
    sqlite_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])
    statement_rows = int(connection.execute("SELECT COUNT(*) FROM financial_statement_fact WHERE source='choice'").fetchone()[0])
    dividend_rows = int(connection.execute("SELECT COUNT(*) FROM dividend_fact WHERE source='choice'").fetchone()[0])
    ingestion_runs_before = int(connection.execute("SELECT COUNT(*) FROM financial_ingestion_run").fetchone()[0])

sqlite_statements = database.query_financial_statement_facts(
    source="choice", symbols=SYMBOLS,
    start_report_date=START_REPORT_DATE, end_report_date=END_REPORT_DATE,
)
sqlite_dividends = database.query_dividend_facts(source="choice", symbols=SYMBOLS)

print("SQLite数据库：", DB_PATH)
print("数据库完整性：", sqlite_integrity)
print("数据库累计财务事实：", statement_rows)
print("数据库累计分红事实：", dividend_rows)
display(sqlite_statements.head(12))
display(sqlite_dividends.head(12))
if sqlite_statements.empty:
    raise RuntimeError("SQLite没有Choice财务事实，请确认QIANJI_DB_PATH指向此前验收数据库。")

SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
数据库完整性： ok
数据库累计财务事实： 60
数据库累计分红事实： 44


,source,symbol,statement_type,report_date,indicator,value_numeric,value_text,currency,unit,fetched_at
0,choice,000001.SZ,balance,2025-12-31,SUMASSET,5.925777e+12,5925777000000.0,CNY,CNY,2026-09-02T05:24:54.155920+00:00
1,choice,000001.SZ,balance,2025-12-31,SUMLIAB,5.374593e+12,5374593000000.0,CNY,CNY,2026-09-02T05:24:54.155920+00:00
2,choice,000001.SZ,balance,2025-12-31,SUMSHEQUITY,5.511840e+11,551184000000.0,CNY,CNY,2026-09-02T05:24:54.155920+00:00
3,choice,000001.SZ,cashflow,2025-12-31,NETFINACASHFLOW,-1.492360e+11,-149236000000.0,CNY,CNY,2026-09-02T05:24:54.971275+00:00
4,choice,000001.SZ,cashflow,2025-12-31,NETINVCASHFLOW,-7.903600e+10,-79036000000.0,CNY,CNY,2026-09-02T05:24:54.971275+00:00
5,choice,000001.SZ,cashflow,2025-12-31,NETOPERATECASHFLOW,3.158580e+11,315858000000.0,CNY,CNY,2026-09-02T05:24:54.971275+00:00
6,choice,000001.SZ,cashflow,2025-12-31,NICASHEQUI,8.568900e+10,85689000000.0,CNY,CNY,2026-09-02T05:24:54.971275+00:00
7,choice,000001.SZ,income,2025-12-31,NETPROFIT,4.263300e+10,42633000000.0,CNY,CNY,2026-09-02T05:31:36.954792+00:00
8,choice,000001.SZ,income,2025-12-31,OPERATEREVE,1.314420e+11,131442000000.0,CNY,CNY,2026-09-02T05:31:36.954792+00:00
9,choice,000001.SZ,income,2025-12-31,PARENTNETPROFIT,4.263300e+10,42633000000.0,CNY,CNY,2026-09-02T05:31:36.954792+00:00


,source,symbol,report_date,indicator,value_numeric,value_text,currency,unit,fetched_at
0,choice,000001.SZ,2025-12-31,DIVCAPITPSRATIO,NaN,None,CNY,vendor_raw_ratio,2026-09-02T05:34:27.037519+00:00
1,choice,000001.SZ,2025-12-31,DIVCASHPSAFTAX,NaN,0.3240 或 0.3600,CNY,CNY/share,2026-09-02T05:34:27.037519+00:00
2,choice,000001.SZ,2025-12-31,DIVCASHPSBFTAX,3.600000e-01,0.36,CNY,CNY/share,2026-09-02T05:34:27.037519+00:00
3,choice,000001.SZ,2025-12-31,DIVEXDATE,NaN,2026-06-12,CNY,date,2026-09-02T05:34:27.037519+00:00
4,choice,000001.SZ,2025-12-31,DIVIMPLANNCDATE,NaN,2026-06-05,CNY,date,2026-09-02T05:34:27.037519+00:00
5,choice,000001.SZ,2025-12-31,DIVPAYDATE,NaN,2026-06-12,CNY,date,2026-09-02T05:34:27.037519+00:00
6,choice,000001.SZ,2025-12-31,DIVRECORDDATE,NaN,2026-06-11,CNY,date,2026-09-02T05:34:27.037519+00:00
7,choice,000001.SZ,2025-12-31,DIVRTISSBASESHARES,1.940592e+06,1940591.8198,CNY,10k_share,2026-09-02T05:34:27.037519+00:00
8,choice,000001.SZ,2025-12-31,DIVSTOCKPSRATIO,NaN,None,CNY,vendor_raw_ratio,2026-09-02T05:34:27.037519+00:00
9,choice,000001.SZ,2025-12-31,DIVWAY,NaN,"10派3.60元(含税,扣税后3.24元)",CNY,text,2026-09-02T05:34:27.037519+00:00


In [5]:
from openbb import obb

provider_routes = dict(getattr(obb.coverage, "providers", {}))
qianji_routes = provider_routes.get("qianji", [])
expected_routes = {
    ".equity.fundamental.income",
    ".equity.fundamental.balance",
    ".equity.fundamental.cash",
    ".equity.fundamental.dividends",
}
missing_routes = sorted(expected_routes - set(qianji_routes))
print("OpenBB发现qianji：", "qianji" in provider_routes)
print("qianji路由：", qianji_routes)
print("缺失路由：", missing_routes)
if missing_routes:
    raise RuntimeError("OpenBB尚未发现新财务路由，请重新运行00号Notebook并彻底重启内核。")

OpenBB发现qianji： True
qianji路由： ['.equity.fundamental.balance', '.equity.fundamental.cash', '.equity.fundamental.dividends', '.equity.fundamental.income', '.equity.price.historical', '.equity.search']
缺失路由： []


In [6]:
query_frames = {}
query_audit = []

def run_query(label, symbol, call):
    try:
        result = call()
        frame = result.to_dataframe().reset_index(drop=True)
        query_frames[(label, symbol)] = frame
        query_audit.append({"dataset": label, "symbol": symbol, "provider": result.provider, "rows": len(frame), "error": ""})
    except Exception as exc:
        query_frames[(label, symbol)] = pd.DataFrame()
        query_audit.append({"dataset": label, "symbol": symbol, "provider": "", "rows": 0, "error": f"{type(exc).__name__}: {str(exc)[:1000]}"})

for symbol in SYMBOLS:
    common = dict(symbol=symbol, start_date=START_REPORT_DATE, end_date=END_REPORT_DATE, limit=LIMIT, provider="qianji", source="choice", use_cache=False)
    run_query("income", symbol, lambda common=common: obb.equity.fundamental.income(**common))
    run_query("balance", symbol, lambda common=common: obb.equity.fundamental.balance(**common))
    run_query("cashflow", symbol, lambda common=common: obb.equity.fundamental.cash(**common))
    dividend_args = dict(symbol=symbol, start_date=START_REPORT_DATE, end_date=END_REPORT_DATE, provider="qianji", source="choice", use_cache=False)
    run_query("dividend", symbol, lambda dividend_args=dividend_args: obb.equity.fundamental.dividends(**dividend_args))

query_audit_df = pd.DataFrame(query_audit)
display(query_audit_df)
for key, frame in query_frames.items():
    print(key, len(frame))
    display(frame)

,dataset,symbol,provider,rows,error
0,income,000001.SZ,qianji,2,
1,balance,000001.SZ,qianji,2,
2,cashflow,000001.SZ,qianji,2,
3,dividend,000001.SZ,qianji,1,
4,income,600519.SH,qianji,2,
5,balance,600519.SH,qianji,2,
6,cashflow,600519.SH,qianji,2,
7,dividend,600519.SH,qianji,1,
8,income,300750.SZ,qianji,2,
9,balance,300750.SZ,qianji,2,


('income', '000001.SZ') 2


,period_ending,fiscal_period,fiscal_year,symbol,reported_currency,source,revenue,consolidated_net_income,net_income_attributable_to_parent
0,2026-06-30,Q2,2026,000001.SZ,CNY,choice,7.061700e+10,2.569600e+10,2.569600e+10
1,2025-12-31,FY,2025,000001.SZ,CNY,choice,1.314420e+11,4.263300e+10,4.263300e+10


('balance', '000001.SZ') 2


,period_ending,fiscal_period,fiscal_year,symbol,reported_currency,source,total_assets,total_liabilities,total_common_equity
0,2026-06-30,Q2,2026,000001.SZ,CNY,choice,6.028785e+12,5.480571e+12,5.482140e+11
1,2025-12-31,FY,2025,000001.SZ,CNY,choice,5.925777e+12,5.374593e+12,5.511840e+11


...

,period_ending,fiscal_period,fiscal_year,symbol,reported_currency,source,total_assets,total_liabilities,total_common_equity
0,2026-06-30,Q2,2026,300750.SZ,CNY,choice,1.138881e+12,7.249256e+11,4.139552e+11
1,2025-12-31,FY,2025,300750.SZ,CNY,choice,9.748275e+11,6.038012e+11,3.710263e+11


('cashflow', '300750.SZ') 2


,period_ending,fiscal_period,fiscal_year,symbol,reported_currency,source,net_cash_from_operating_activities,net_cash_from_investing_activities,net_cash_from_financing_activities,net_change_in_cash_and_equivalents
0,2026-06-30,Q2,2026,300750.SZ,CNY,choice,6.021685e+10,-3.692909e+10,2.431026e+10,4.065648e+10
1,2025-12-31,FY,2025,300750.SZ,CNY,choice,1.332200e+11,-9.447579e+10,-6.309543e+09,2.977001e+10


('dividend', '300750.SZ') 2


,symbol,ex_dividend_date,amount,report_date,declaration_date,record_date,payment_date,amount_after_tax_text,dividend_plan,share_base_10k
0,300750.SZ,2026-08-10,1.411,2026-06-30,2026-08-04,2026-08-07,2026-08-10,1.2699 或 1.4110,"10派14.11元(含税,扣税后12.699元)",438003.5575
1,300750.SZ,2026-04-22,6.957,2025-12-31,2026-04-15,2026-04-21,2026-04-22,6.2613 或 6.9570,"10派69.57元(含税,扣税后62.613元)",437618.2522


In [7]:
FIELD_MAPS = {
    "income": {"OPERATEREVE": "revenue", "NETPROFIT": "consolidated_net_income", "PARENTNETPROFIT": "net_income_attributable_to_parent"},
    "balance": {"SUMASSET": "total_assets", "SUMLIAB": "total_liabilities", "SUMSHEQUITY": "total_common_equity"},
    "cashflow": {"NETOPERATECASHFLOW": "net_cash_from_operating_activities", "NETINVCASHFLOW": "net_cash_from_investing_activities", "NETFINACASHFLOW": "net_cash_from_financing_activities", "NICASHEQUI": "net_change_in_cash_and_equivalents"},
}

reconciliation_rows = []
for dataset, field_map in FIELD_MAPS.items():
    source_type = "cashflow" if dataset == "cashflow" else dataset
    scoped = sqlite_statements[sqlite_statements["statement_type"] == source_type]
    for symbol in SYMBOLS:
        frame = query_frames[(dataset, symbol)]
        if frame.empty:
            continue
        for _, row in frame.iterrows():
            period = str(row["period_ending"])[:10]
            for indicator, openbb_field in field_map.items():
                matched = scoped[(scoped["symbol"] == symbol) & (scoped["report_date"].astype(str) == period) & (scoped["indicator"] == indicator)]
                sqlite_value = None if matched.empty else matched.iloc[0]["value_numeric"]
                openbb_value = row.get(openbb_field)
                difference = None if pd.isna(sqlite_value) or pd.isna(openbb_value) else float(openbb_value) - float(sqlite_value)
                reconciliation_rows.append({"dataset": dataset, "symbol": symbol, "report_date": period, "indicator": indicator, "openbb_field": openbb_field, "sqlite_value": sqlite_value, "openbb_value": openbb_value, "difference": difference})

reconciliation_columns = ["dataset", "symbol", "report_date", "indicator", "openbb_field", "sqlite_value", "openbb_value", "difference"]
statement_reconciliation = pd.DataFrame(reconciliation_rows, columns=reconciliation_columns)
statement_reconciliation["matched"] = statement_reconciliation["difference"].fillna(0).abs() <= 0.01 if not statement_reconciliation.empty else pd.Series(dtype=bool)
display(statement_reconciliation)

,dataset,symbol,report_date,indicator,openbb_field,sqlite_value,openbb_value,difference,matched
0,income,000001.SZ,2026-06-30,OPERATEREVE,revenue,7.061700e+10,7.061700e+10,0.0,True
1,income,000001.SZ,2026-06-30,NETPROFIT,consolidated_net_income,2.569600e+10,2.569600e+10,0.0,True
2,income,000001.SZ,2026-06-30,PARENTNETPROFIT,net_income_attributable_to_parent,2.569600e+10,2.569600e+10,0.0,True
3,income,000001.SZ,2025-12-31,OPERATEREVE,revenue,1.314420e+11,1.314420e+11,0.0,True
4,income,000001.SZ,2025-12-31,NETPROFIT,consolidated_net_income,4.263300e+10,4.263300e+10,0.0,True
5,income,000001.SZ,2025-12-31,PARENTNETPROFIT,net_income_attributable_to_parent,4.263300e+10,4.263300e+10,0.0,True
6,income,600519.SH,2026-06-30,OPERATEREVE,revenue,9.070326e+10,9.070326e+10,0.0,True
7,income,600519.SH,2026-06-30,NETPROFIT,consolidated_net_income,4.603333e+10,4.603333e+10,0.0,True
8,income,600519.SH,2026-06-30,PARENTNETPROFIT,net_income_attributable_to_parent,4.451688e+10,4.451688e+10,0.0,True
9,income,600519.SH,2025-12-31,OPERATEREVE,revenue,1.688381e+11,1.688381e+11,0.0,True


In [8]:
from datetime import datetime

def normalize_choice_date(value):
    text = str(value).strip()[:10]
    for pattern in ("%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%m/%d/%y"):
        try:
            return datetime.strptime(text, pattern).date().isoformat()
        except ValueError:
            continue
    raise ValueError(f"无法识别Choice日期：{text!r}")

dividend_expected_rows = []
for (symbol, report_date), group in sqlite_dividends.groupby(["symbol", "report_date"]):
    facts = group.set_index("indicator")
    if "DIVEXDATE" not in facts.index or "DIVCASHPSBFTAX" not in facts.index:
        continue
    ex_date = facts.loc["DIVEXDATE", "value_text"]
    amount = facts.loc["DIVCASHPSBFTAX", "value_numeric"]
    if pd.isna(ex_date) or pd.isna(amount):
        continue
    ex_date_iso = normalize_choice_date(ex_date)
    if START_REPORT_DATE <= ex_date_iso <= END_REPORT_DATE:
        dividend_expected_rows.append({"symbol": symbol, "report_date": str(report_date)[:10], "ex_dividend_date": ex_date_iso, "amount": float(amount)})
dividend_expected = pd.DataFrame(dividend_expected_rows)
dividend_openbb = pd.concat([frame.assign(request_symbol=symbol) for (dataset, symbol), frame in query_frames.items() if dataset == "dividend" and not frame.empty], ignore_index=True) if any(dataset == "dividend" and not frame.empty for (dataset, _), frame in query_frames.items()) else pd.DataFrame()

if not dividend_expected.empty and not dividend_openbb.empty:
    dividend_openbb["report_date"] = dividend_openbb["report_date"].astype(str).str[:10]
    dividend_openbb["ex_dividend_date"] = dividend_openbb["ex_dividend_date"].astype(str).str[:10]
    dividend_reconciliation = dividend_expected.merge(dividend_openbb[["symbol", "report_date", "ex_dividend_date", "amount"]], on=["symbol", "report_date", "ex_dividend_date"], how="outer", suffixes=("_sqlite", "_openbb"), indicator=True)
    dividend_reconciliation["difference"] = dividend_reconciliation["amount_openbb"] - dividend_reconciliation["amount_sqlite"]
else:
    dividend_reconciliation = pd.DataFrame()
display(dividend_expected)
display(dividend_openbb)
display(dividend_reconciliation)

,symbol,report_date,ex_dividend_date,amount
0,000001.SZ,2025-12-31,2026-06-12,0.36000
1,300750.SZ,2025-12-31,2026-04-22,6.95700
2,300750.SZ,2026-06-30,2026-08-10,1.41100
3,600519.SH,2025-12-31,2026-06-26,28.02423


,symbol,ex_dividend_date,amount,report_date,declaration_date,record_date,payment_date,amount_after_tax_text,dividend_plan,share_base_10k,request_symbol
0,000001.SZ,2026-06-12,0.36000,2025-12-31,2026-06-05,2026-06-11,2026-06-12,0.3240 或 0.3600,"10派3.60元(含税,扣税后3.24元)",1.940592e+06,000001.SZ
1,600519.SH,2026-06-26,28.02423,2025-12-31,2026-06-22,2026-06-25,2026-06-26,25.2218 或 28.0242,"10派280.2423元(含税,扣税后252.2181元)",1.250082e+05,600519.SH
2,300750.SZ,2026-08-10,1.41100,2026-06-30,2026-08-04,2026-08-07,2026-08-10,1.2699 或 1.4110,"10派14.11元(含税,扣税后12.699元)",4.380036e+05,300750.SZ
3,300750.SZ,2026-04-22,6.95700,2025-12-31,2026-04-15,2026-04-21,2026-04-22,6.2613 或 6.9570,"10派69.57元(含税,扣税后62.613元)",4.376183e+05,300750.SZ


,symbol,report_date,ex_dividend_date,amount_sqlite,amount_openbb,_merge,difference
0,000001.SZ,2025-12-31,2026-06-12,0.36000,0.36000,both,0.0
1,300750.SZ,2025-12-31,2026-04-22,6.95700,6.95700,both,0.0
2,300750.SZ,2026-06-30,2026-08-10,1.41100,1.41100,both,0.0
3,600519.SH,2025-12-31,2026-06-26,28.02423,28.02423,both,0.0


In [9]:
with database.connect() as connection:
    ingestion_runs_after = int(connection.execute("SELECT COUNT(*) FROM financial_ingestion_run").fetchone()[0])

quality_rows = []
def gate(name, passed, evidence):
    quality_rows.append({"check": name, "status": "PASS" if passed else "FAIL", "evidence": str(evidence)})

all_qianji = bool(len(query_audit_df)) and query_audit_df.loc[query_audit_df["error"] == "", "provider"].eq("qianji").all()
query_errors = query_audit_df[query_audit_df["error"] != ""]
statement_match = not statement_reconciliation.empty and statement_reconciliation["matched"].all()
dividend_match = (dividend_expected.empty and dividend_openbb.empty) or (not dividend_reconciliation.empty and dividend_reconciliation["_merge"].eq("both").all() and dividend_reconciliation["difference"].fillna(0).abs().le(1e-12).all())

gate("qianji-data-mini版本>=0.9.0", version_ok, installed_qianji)
gate("SQLite完整性", sqlite_integrity == "ok", sqlite_integrity)
gate("四个OpenBB路由已注册", not missing_routes, missing_routes)
gate("所有查询无错误", query_errors.empty, query_errors.to_dict("records"))
gate("成功结果均来自qianji", all_qianji, query_audit_df[["dataset", "symbol", "provider"]].to_dict("records"))
gate("三张报表与SQLite逐项一致", statement_match, f"rows={len(statement_reconciliation)}")
gate("现金分红与SQLite一致", dividend_match, f"sqlite={len(dividend_expected)}, openbb={len(dividend_openbb)}")
gate("查询未新增采集日志", ingestion_runs_before == ingestion_runs_after, f"before={ingestion_runs_before}, after={ingestion_runs_after}")
gate("Notebook不导入EmQuantAPI", "EmQuantAPI" not in sys.modules, "EmQuantAPI" in sys.modules)

quality_gates = pd.DataFrame(quality_rows)
failed_count = int((quality_gates["status"] == "FAIL").sum())
display(quality_gates)
print("通过：", len(quality_gates) - failed_count, "失败：", failed_count)

,check,status,evidence
0,qianji-data-mini版本>=0.9.0,PASS,0.9.0
1,SQLite完整性,PASS,ok
2,四个OpenBB路由已注册,PASS,[]
3,所有查询无错误,PASS,[]
4,成功结果均来自qianji,PASS,"[{'dataset': 'income', 'symbol': '000001.SZ', ..."
5,三张报表与SQLite逐项一致,PASS,rows=60
6,现金分红与SQLite一致,PASS,"sqlite=4, openbb=4"
7,查询未新增采集日志,PASS,"before=6, after=6"
8,Notebook不导入EmQuantAPI,PASS,False


通过： 9 失败： 0


In [10]:
data_map = pd.DataFrame([
    {"sqlite_table": "financial_statement_fact", "openbb_model": "IncomeStatement", "openbb_route": "obb.equity.fundamental.income", "provider": "qianji", "vendor_api_called": False},
    {"sqlite_table": "financial_statement_fact", "openbb_model": "BalanceSheet", "openbb_route": "obb.equity.fundamental.balance", "provider": "qianji", "vendor_api_called": False},
    {"sqlite_table": "financial_statement_fact", "openbb_model": "CashFlowStatement", "openbb_route": "obb.equity.fundamental.cash", "provider": "qianji", "vendor_api_called": False},
    {"sqlite_table": "dividend_fact", "openbb_model": "HistoricalDividends", "openbb_route": "obb.equity.fundamental.dividends", "provider": "qianji", "vendor_api_called": False},
])
display(data_map)

,sqlite_table,openbb_model,openbb_route,provider,vendor_api_called
0,financial_statement_fact,IncomeStatement,obb.equity.fundamental.income,qianji,False
1,financial_statement_fact,BalanceSheet,obb.equity.fundamental.balance,qianji,False
2,financial_statement_fact,CashFlowStatement,obb.equity.fundamental.cash,qianji,False
3,dividend_fact,HistoricalDividends,obb.equity.fundamental.dividends,qianji,False


In [11]:
import json
from datetime import datetime, timezone
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill

timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice财务分红_OpenBB查询验收_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice财务分红_OpenBB查询验收_{timestamp}.json"
overview = pd.DataFrame([
    ["generated_at", datetime.now(timezone.utc).isoformat()],
    ["python", sys.executable], ["database", str(DB_PATH)],
    ["qianji_data_mini_version", installed_qianji], ["openbb_version", installed_openbb],
    ["provider", "qianji"], ["original_source", "choice"],
    ["choice_api_called", False], ["passed", len(quality_gates) - failed_count], ["failed", failed_count],
], columns=["item", "value"])

sheets = {"验收概览": overview, "质量门槛": quality_gates, "查询审计": query_audit_df, "财务逐项核对": statement_reconciliation, "分红SQLite预期": dividend_expected, "分红OpenBB结果": dividend_openbb, "分红核对": dividend_reconciliation, "SQLite财务事实": sqlite_statements, "SQLite分红事实": sqlite_dividends, "数据地图": data_map}
for (dataset, symbol), frame in query_frames.items():
    sheets[f"{dataset}_{symbol.replace('.', '_')}"] = frame

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
workbook = load_workbook(excel_path)
for worksheet in workbook.worksheets:
    for cell in worksheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="2F75B5")
        cell.alignment = Alignment(horizontal="center")
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for column in worksheet.columns:
        width = min(45, max(10, max(len(str(cell.value or "")) for cell in column) + 2))
        worksheet.column_dimensions[column[0].column_letter].width = width
workbook.save(excel_path)

payload = {"generated_at": datetime.now(timezone.utc).isoformat(), "database": str(DB_PATH), "version": installed_qianji, "choice_api_called": False, "quality_gates": quality_gates.to_dict("records"), "query_audit": query_audit_df.to_dict("records")}
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Excel证据：", excel_path)
print("JSON证据：", json_path)

Excel证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice财务分红_OpenBB查询验收_20260902_142155.xlsx
JSON证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice财务分红_OpenBB查询验收_20260902_142155.json


In [12]:
if failed_count == 0:
    print("✅ 10号验收通过：SQLite财务与分红已经能够通过OpenBB qianji Provider查询。")
    print("本Notebook未调用Choice，不消耗CTR流量。")
else:
    print(f"⚠️ 10号验收存在{failed_count}项失败，请查看质量门槛和查询审计。")
    display(quality_gates[quality_gates["status"] == "FAIL"])
if STRICT_MODE and failed_count:
    raise RuntimeError(f"10号验收存在{failed_count}项失败。")

✅ 10号验收通过：SQLite财务与分红已经能够通过OpenBB qianji Provider查询。
本Notebook未调用Choice，不消耗CTR流量。
